# 01 — Data Cleaning (Diabites)

Ce notebook reprend **étape par étape** les actions du script `src/preprocessing/cleaning.py` (classe `DataCleaner`), avec pour chaque étape :

- **Action** : ce que fait le code
- **Justification** : pourquoi on le fait, et ce que montrent les résultats

**Ordre du pipeline :** chargement → inspection → nettoyage du texte → conversion des types → valeurs manquantes → outliers → doublons → validation → sauvegarde.

## 0. Imports et chemins

In [16]:
import pandas as pd
import numpy as np

RAW_PATH = "../data/raw/dataset.csv"
CLEAN_PATH = "../data/processed/dataset_clean.csv"

## 1. Chargement du dataset

**Action :** lecture du CSV brut avec `pd.read_csv`.


In [4]:
df = pd.read_csv(RAW_PATH)
df.head()

,Unnamed: 0,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
0,0,6,148,72,35,0,33.6,0.627,50
1,1,1,85,66,29,0,26.6,0.351,31
2,2,8,183,64,0,0,23.3,0.672,32
3,3,1,89,66,23,94,28.1,0.167,21
4,4,0,137,40,35,168,43.1,2.288,33


## 2. STRUCTURE DU DATASET

**Action :** Aficher le structure du dataset


In [5]:
print("Dimensions :", df.shape)

print("\nColonnes :")
print(df.columns.tolist())

print("\nTypes et valeurs non nulles :")
df.info()

print("\nStatistiques numériques :")
print(df.describe())

Dimensions : (768, 9)

Colonnes :
['Unnamed: 0', 'Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']

Types et valeurs non nulles :
<class 'pandas.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Unnamed: 0                768 non-null    int64  
 1   Pregnancies               768 non-null    int64  
 2   Glucose                   768 non-null    int64  
 3   BloodPressure             768 non-null    int64  
 4   SkinThickness             768 non-null    int64  
 5   Insulin                   768 non-null    int64  
 6   BMI                       768 non-null    float64
 7   DiabetesPedigreeFunction  768 non-null    float64
 8   Age                       768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB

Statistiques numériques :
       Unnamed: 0  Pregnancies 

## 3. VALEURS MANQUANTES 
**Action :** afficher les valeurs manquantes


In [6]:
missing = df.isnull().sum()

print(missing[missing > 0].sort_values(ascending=False))

Series([], dtype: int64)


## 4 OUTLIERS
**Action :** aficher le numbre outliers in all column nimerique

## 5. DOUBLONS

**Action :** aficher le number DOUBLONS

In [8]:
duplicates = df.duplicated().sum()

print("\n DOUBLONS ")
print("Nombre de doublons :", duplicates)


 DOUBLONS 
Nombre de doublons : 0


## categorical values

In [9]:
categorical_columns = df.select_dtypes(include="object").columns

for col in categorical_columns:
    print(f"\n===== {col} =====")
    print(df[col].value_counts(dropna=False).head(20))

## Remove WHITE SPACE from columns

In [10]:
df.columns = df.columns.str.strip()

for col in df.select_dtypes("object"):
    df[col] = df[col].str.strip()

df = df.replace(["NaN", "nan", ""], pd.NA)

## Determine outliers

In [13]:
numeric_cols = df.select_dtypes(include="number").columns

print(numeric_cols)

Index(['Unnamed: 0', 'Pregnancies', 'Glucose', 'BloodPressure',
       'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age'],
      dtype='str')


In [14]:
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    
    IQR = Q3 - Q1
    
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    
    print(f"\n===== {col} =====")
    print(f"Q1       : {Q1:.2f}")
    print(f"Q3       : {Q3:.2f}")
    print(f"IQR      : {IQR:.2f}")
    print(f"Lower    : {lower:.2f}")
    print(f"Upper    : {upper:.2f}")
    print(f"Outliers : {outliers}")


===== Unnamed: 0 =====
Q1       : 191.75
Q3       : 575.25
IQR      : 383.50
Lower    : -383.50
Upper    : 1150.50
Outliers : 0

===== Pregnancies =====
Q1       : 1.00
Q3       : 6.00
IQR      : 5.00
Lower    : -6.50
Upper    : 13.50
Outliers : 4

===== Glucose =====
Q1       : 99.00
Q3       : 140.25
IQR      : 41.25
Lower    : 37.12
Upper    : 202.12
Outliers : 5

===== BloodPressure =====
Q1       : 62.00
Q3       : 80.00
IQR      : 18.00
Lower    : 35.00
Upper    : 107.00
Outliers : 45

===== SkinThickness =====
Q1       : 0.00
Q3       : 32.00
IQR      : 32.00
Lower    : -48.00
Upper    : 80.00
Outliers : 1

===== Insulin =====
Q1       : 0.00
Q3       : 127.25
IQR      : 127.25
Lower    : -190.88
Upper    : 318.12
Outliers : 34

===== BMI =====
Q1       : 27.30
Q3       : 36.60
IQR      : 9.30
Lower    : 13.35
Upper    : 50.55
Outliers : 19

===== DiabetesPedigreeFunction =====
Q1       : 0.24
Q3       : 0.63
IQR      : 0.38
Lower    : -0.33
Upper    : 1.20
Outliers : 29

=====

## Repalce the 0 values with NAN

In [25]:
cols_missing = [
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI"
]

df[cols_missing] = df[cols_missing].replace(0, np.nan)
    
df.info()
df[cols_missing].isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Unnamed: 0                768 non-null    int64  
 1   Pregnancies               768 non-null    int64  
 2   Glucose                   763 non-null    float64
 3   BloodPressure             733 non-null    float64
 4   SkinThickness             541 non-null    float64
 5   Insulin                   394 non-null    float64
 6   BMI                       757 non-null    float64
 7   DiabetesPedigreeFunction  768 non-null    float64
 8   Age                       768 non-null    int64  
dtypes: float64(6), int64(3)
memory usage: 54.1 KB


Glucose            5
BloodPressure     35
SkinThickness    227
Insulin          374
BMI               11
dtype: int64

## export result

In [26]:
result = df.to_csv(CLEAN_PATH, index=False)